# 🌿 Black Pepper Disease Detection - Complete Training Guide
# Target: 90%+ Accuracy

**Train a high-accuracy CNN model for Black Pepper disease detection using FREE Google Colab GPU!**

## ✨ What's Different in This Notebook:
- ✅ **Strong regularization** to prevent overfitting
- ✅ **Class weight balancing** for imbalanced datasets
- ✅ **Progressive training strategy** (3 stages)
- ✅ **Advanced data augmentation**
- ✅ **Data quality checks** before training
- ✅ **Automatic best model selection**

## 📋 Steps to Follow:
1. **Upload this notebook** to Google Colab
2. **Enable GPU**: Runtime → Change runtime type → GPU (T4)
3. **Upload your dataset** to Google Drive with this structure:
   ```
   MyDrive/
     └── black_pepper_dataset/
           ├── Healthy/          (healthy leaf images)
           ├── Disease1/         (disease 1 images)
           ├── Disease2/         (disease 2 images)
           └── Disease3/         (disease 3 images)
   ```
4. **Run all cells** from top to bottom
5. **Download trained model** at the end

---

## 📍 STEP 1: Check GPU & Install Dependencies

In [ ]:
import tensorflow as tf
import os

print("="*70)
print("🔍 SYSTEM CHECK")
print("="*70)

print(f"\n📦 TensorFlow Version: {tf.__version__}")

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if len(gpus) > 0:
    print(f"✅ GPU is ENABLED: {gpus[0].name}")
    print("   Training will be FAST! 🚀")
    
    # Set memory growth to avoid OOM errors
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("   Memory growth enabled (prevents OOM errors)")
else:
    print("⚠️ WARNING: GPU NOT ENABLED!")
    print("   → Go to: Runtime → Change runtime type → Select 'GPU'")
    print("   → Then restart runtime and run this cell again")
    print("\n   Training without GPU will be 10-20x slower! ⏰")

print("\n" + "="*70)
print("✅ Ready to proceed!")
print("="*70)

## 📍 STEP 2: Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
print("\n✅ Google Drive mounted successfully!")

## 📍 STEP 3: Configure Your Dataset Path

**IMPORTANT**: Update the path below to match YOUR Google Drive folder structure!

In [ ]:
import os
import shutil
from collections import Counter

# ============================================================================
# 🎯 CONFIGURE THIS: Update to match your Google Drive folder path
# ============================================================================
DRIVE_DATASET_PATH = '/content/drive/MyDrive/black_pepper_dataset'

# You can also specify a direct path to the folder containing class folders:
# DRIVE_DATASET_PATH = '/content/drive/MyDrive/my_pepper_images'

print("="*70)
print("🔍 SEARCHING FOR YOUR DATASET...")
print("="*70)

print(f"\n📂 Looking at: {DRIVE_DATASET_PATH}")

if not os.path.exists(DRIVE_DATASET_PATH):
    print(f"\n❌ ERROR: Folder not found!")
    print(f"   Path checked: {DRIVE_DATASET_PATH}")
    print("\n📝 SOLUTION:")
    print("   1. Check your Google Drive folder name (case-sensitive!)")
    print("   2. Update DRIVE_DATASET_PATH variable above")
    print("   3. Re-run this cell")
    raise FileNotFoundError(f"Dataset folder not found: {DRIVE_DATASET_PATH}")

# List all contents
contents = os.listdir(DRIVE_DATASET_PATH)
print(f"\n📁 Found {len(contents)} items in folder:")

folders = []
for item in contents:
    item_path = os.path.join(DRIVE_DATASET_PATH, item)
    if os.path.isdir(item_path):
        folders.append(item)
        print(f"   📁 {item}/")
    else:
        print(f"   📄 {item}")

if not folders:
    print("\n❌ ERROR: No subfolders found!")
    print("   Your dataset should have class folders (e.g., Healthy, Disease1, etc.)")
    raise ValueError("No class folders found in dataset directory")

print(f"\n✅ Found {len(folders)} potential class folders")
print("   Proceeding to analyze dataset structure...")

## 📍 STEP 4: Analyze & Prepare Dataset

This cell will:
- Count images in each class
- Check for data quality issues
- Copy dataset to local storage for faster training
- Calculate class weights for imbalanced data

In [ ]:
import os
import shutil
from collections import Counter

print("="*70)
print("📊 DATASET ANALYSIS")
print("="*70)

# Analyze dataset structure
dataset_info = {}
total_images = 0
valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.JPEG', '.PNG', '.BMP'}

print(f"\n🔍 Analyzing images in each class folder...\n")

for folder in folders:
    folder_path = os.path.join(DRIVE_DATASET_PATH, folder)
    
    # Count valid images
    image_files = [f for f in os.listdir(folder_path) 
                   if os.path.splitext(f)[1] in valid_extensions]
    
    count = len(image_files)
    dataset_info[folder] = count
    total_images += count
    
    # Show status with icon
    if count == 0:
        icon = "❌"
    elif count < 100:
        icon = "⚠️"
    elif count < 300:
        icon = "✅"
    else:
        icon = "🌟"
    
    print(f"   {icon} {folder:30s}: {count:5d} images")

print("\n" + "-"*70)
print(f"   📊 TOTAL IMAGES: {total_images}")
print("-"*70)

# Data quality checks
print("\n🔍 DATA QUALITY CHECKS:")
print("-"*70)

issues_found = []

# Check 1: Minimum images per class
min_images = 100
low_count_classes = [cls for cls, cnt in dataset_info.items() if cnt < min_images]
if low_count_classes:
    print(f"⚠️  Warning: {len(low_count_classes)} class(es) have < {min_images} images")
    for cls in low_count_classes:
        print(f"   • {cls}: {dataset_info[cls]} images")
    print(f"   Recommendation: Add more images (target: 300+ per class)")
    issues_found.append("low_image_count")
else:
    print(f"✅ All classes have ≥ {min_images} images")

# Check 2: Class imbalance
if dataset_info:
    max_count = max(dataset_info.values())
    min_count = min(dataset_info.values())
    imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
    
    print(f"\n📊 Class Imbalance Ratio: {imbalance_ratio:.2f}:1")
    if imbalance_ratio > 3.0:
        print(f"⚠️  High imbalance detected!")
        print(f"   • Largest class: {max_count} images")
        print(f"   • Smallest class: {min_count} images")
        print(f"   → Will use class weights to balance training")
        issues_found.append("high_imbalance")
    elif imbalance_ratio > 1.5:
        print(f"⚠️  Moderate imbalance detected")
        print(f"   → Will use class weights to balance training")
        issues_found.append("moderate_imbalance")
    else:
        print(f"✅ Dataset is well-balanced")

# Check 3: Minimum total images
if total_images < 500:
    print(f"\n⚠️  Total dataset size is small ({total_images} images)")
    print(f"   → May affect model accuracy")
    print(f"   → Recommendation: Collect 1000+ images total")
    issues_found.append("small_dataset")
elif total_images < 1000:
    print(f"\n✅ Dataset size is moderate ({total_images} images)")
    print(f"   → Should achieve good results")
else:
    print(f"\n🌟 Dataset size is good ({total_images} images)")
    print(f"   → Should achieve excellent results!")

# Check 4: Number of classes
num_classes = len(dataset_info)
if num_classes < 2:
    print(f"\n❌ ERROR: Only {num_classes} class found!")
    print(f"   You need at least 2 classes for classification")
    raise ValueError("Insufficient number of classes")
else:
    print(f"\n✅ Number of classes: {num_classes}")

print("\n" + "="*70)

if not issues_found:
    print("🎉 PERFECT! No major data quality issues detected!")
elif len(issues_found) <= 2:
    print("✅ Dataset is usable with minor issues (will be handled automatically)")
else:
    print("⚠️  Multiple data quality issues detected")
    print("   → Training will proceed but accuracy may be affected")
    print("   → Consider improving dataset quality for best results")

print("="*70)

# Copy dataset to local storage for faster training
print("\n📋 Copying dataset to local storage for faster training...")
print("   (This may take 1-3 minutes depending on dataset size)")

LOCAL_DATASET = '/content/black_pepper_dataset'

if os.path.exists(LOCAL_DATASET):
    shutil.rmtree(LOCAL_DATASET)

os.makedirs(LOCAL_DATASET, exist_ok=True)
train_dir = os.path.join(LOCAL_DATASET, 'train')
os.makedirs(train_dir, exist_ok=True)

# Copy each class folder
for folder in folders:
    src = os.path.join(DRIVE_DATASET_PATH, folder)
    dst = os.path.join(train_dir, folder)
    shutil.copytree(src, dst)
    print(f"   ✅ Copied {folder}")

print("\n✅ Dataset ready for training! 🚀")
print(f"📂 Local path: {train_dir}")

## 📍 STEP 5: Create Data Generators with Strong Augmentation

Data augmentation helps the model generalize better and prevents overfitting.

In [ ]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

print("="*70)
print("🎨 CREATING DATA GENERATORS")
print("="*70)

# Image parameters
IMG_SIZE = 224
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2

# Strong data augmentation to prevent overfitting
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,           # Rotate up to 40 degrees
    width_shift_range=0.3,       # Shift horizontally up to 30%
    height_shift_range=0.3,      # Shift vertically up to 30%
    shear_range=0.3,             # Shear transformation
    zoom_range=0.3,              # Zoom in/out up to 30%
    horizontal_flip=True,        # Random horizontal flip
    vertical_flip=True,          # Random vertical flip
    brightness_range=[0.7, 1.3], # Vary brightness
    fill_mode='nearest',         # Fill strategy for transformations
    validation_split=VALIDATION_SPLIT
)

# Validation data (no augmentation, only rescaling)
validation_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT
)

train_dir = f'{LOCAL_DATASET}/train'

print(f"\n📂 Loading images from: {train_dir}")
print(f"📐 Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"📦 Batch size: {BATCH_SIZE}")
print(f"📊 Validation split: {int(VALIDATION_SPLIT*100)}%\n")

# Training generator
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

# Validation generator
validation_generator = validation_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

# Get class information
class_indices = train_generator.class_indices
num_classes = len(class_indices)
class_names = list(class_indices.keys())

print("\n" + "="*70)
print("✅ DATA GENERATORS CREATED")
print("="*70)

print(f"\n📊 Dataset Split:")
print(f"   Training samples: {train_generator.samples}")
print(f"   Validation samples: {validation_generator.samples}")
print(f"   Number of classes: {num_classes}")

print(f"\n🏷️  Classes detected:")
for class_name, class_idx in sorted(class_indices.items(), key=lambda x: x[1]):
    print(f"   {class_idx}. {class_name}")

# Calculate class weights for imbalanced datasets
print("\n⚖️  Calculating class weights...")

# Get class distribution from training generator
class_counts = Counter()
for i in range(len(train_generator.classes)):
    class_counts[train_generator.classes[i]] += 1

# Compute balanced class weights
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

class_weights = {i: weight for i, weight in enumerate(class_weights_array)}

print("\n📊 Class weights (higher = more emphasis during training):")
for class_name, class_idx in sorted(class_indices.items(), key=lambda x: x[1]):
    count = class_counts[class_idx]
    weight = class_weights[class_idx]
    print(f"   {class_name:30s}: {count:4d} images → weight={weight:.3f}")

print("\n✅ Ready for model creation!")

## 📍 STEP 6: Visualize Sample Images (Optional)

Check that images are loading correctly and augmentation is working.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("Generating sample augmented images...\n")

# Get a batch of images
sample_batch = next(train_generator)
images, labels = sample_batch

# Plot 9 sample images
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle('Sample Augmented Training Images', fontsize=16)

for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i])
        
        # Get class name from one-hot encoded label
        class_idx = np.argmax(labels[i])
        class_name = class_names[class_idx]
        
        ax.set_title(f'{class_name}', fontsize=12)
        ax.axis('off')
    else:
        ax.axis('off')

plt.tight_layout()
plt.show()

print("✅ If images look good, proceed to next step!")
print("⚠️  If images are corrupted or wrong, check your dataset folder structure")

## 📍 STEP 7: Build CNN Model with Strong Regularization

Using **MobileNetV2** transfer learning with aggressive regularization to prevent overfitting.

### Key Features for 90% Accuracy:
- ✅ Pre-trained MobileNetV2 base (ImageNet weights)
- ✅ 2× Batch Normalization layers
- ✅ 2× Dropout layers (0.5 rate)
- ✅ L2 kernel regularization
- ✅ Progressive training strategy

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2

print("="*70)
print("🏗️  BUILDING CNN MODEL")
print("="*70)

# Clear any previous models
keras.backend.clear_session()

# Load pre-trained MobileNetV2 (without top classification layer)
print("\n📦 Loading MobileNetV2 base (pre-trained on ImageNet)...")

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model initially (will unfreeze later for fine-tuning)
base_model.trainable = False

print(f"✅ Base model loaded: {len(base_model.layers)} layers")

# Build full model with strong regularization
print("\n🔨 Building custom classification head...")

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# Base model
x = base_model(inputs, training=False)

# Classification head with STRONG regularization
x = layers.GlobalAveragePooling2D()(x)

# First dense block
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)  # High dropout to prevent overfitting
x = layers.Dense(
    256, 
    activation='relu',
    kernel_regularizer=regularizers.l2(0.01)  # L2 regularization
)(x)

# Second dense block
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)  # Second dropout layer
x = layers.Dense(
    128,
    activation='relu',
    kernel_regularizer=regularizers.l2(0.01)
)(x)

# Output layer
outputs = layers.Dense(num_classes, activation='softmax')(x)

# Create model
model = keras.Model(inputs, outputs)

# Compile with low learning rate
INITIAL_LEARNING_RATE = 0.0005

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Print model summary
print("\n" + "="*70)
print("📋 MODEL ARCHITECTURE")
print("="*70)

print(f"\n🔢 Model Statistics:")
print(f"   Total layers: {len(model.layers)}")
print(f"   Trainable layers: {sum([1 for layer in model.layers if layer.trainable])}")
print(f"   Total parameters: {model.count_params():,}")
print(f"   Trainable parameters: {sum([tf.size(var).numpy() for var in model.trainable_variables]):,}")

print(f"\n🎯 Output layer:")
print(f"   Classes: {num_classes}")
print(f"   Activation: softmax")

print(f"\n⚙️  Optimizer: Adam")
print(f"   Initial learning rate: {INITIAL_LEARNING_RATE}")

print("\n✅ Model ready for training!")

## 📍 STEP 8: Train Model (3-Stage Progressive Training)

### Training Strategy for 90% Accuracy:

**Stage 1** (20 epochs): Train classification head only  
**Stage 2** (30 epochs): Fine-tune with partial base unfreezing  
**Stage 3** (40 epochs): Full fine-tuning with very low learning rate  

This progressive approach prevents overfitting while maximizing accuracy.

⏱️ **Expected Time**: 40-60 minutes with GPU (2-4 hours without GPU)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from datetime import datetime

print("="*70)
print("🚀 STARTING 3-STAGE PROGRESSIVE TRAINING")
print("="*70)
print("\nTarget: 90%+ validation accuracy")
print("Strategy: Progressive unfreezing + decreasing learning rates\n")
print("="*70)

# Track all training history
all_histories = []

# ============================================================================
# 🎯 STAGE 1: Train Classification Head Only (Base Frozen)
# ============================================================================
print("\n" + "🔴"*35)
print("📍 STAGE 1: Training Classification Head (Base Frozen)")
print("🔴"*35)

print("\n⚙️  Configuration:")
print(f"   Epochs: 20")
print(f"   Learning rate: {INITIAL_LEARNING_RATE}")
print(f"   Base model: FROZEN")
print(f"   Trainable: Classification head only")
print(f"   Using class weights: YES")

base_model.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_stage1 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1,
        mode='max',
        min_delta=0.001
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=5,
        min_lr=0.00001,
        verbose=1,
        mode='max'
    )
]

print("\n🏃 Training...")
start_time = datetime.now()

history_stage1 = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=20,
    callbacks=callbacks_stage1,
    class_weight=class_weights,
    verbose=1
)

elapsed = datetime.now() - start_time
print(f"\n⏱️  Stage 1 completed in {elapsed}")

best_val_acc_s1 = max(history_stage1.history['val_accuracy']) * 100
final_train_acc_s1 = history_stage1.history['accuracy'][-1] * 100
final_val_acc_s1 = history_stage1.history['val_accuracy'][-1] * 100

print(f"\n📊 Stage 1 Results:")
print(f"   Best validation accuracy: {best_val_acc_s1:.2f}%")
print(f"   Final training accuracy: {final_train_acc_s1:.2f}%")
print(f"   Final validation accuracy: {final_val_acc_s1:.2f}%")
print(f"   Overfitting gap: {abs(final_train_acc_s1 - final_val_acc_s1):.2f}%")

all_histories.append(history_stage1)

# ============================================================================
# 🎯 STAGE 2: Partial Fine-Tuning (Unfreeze Top Layers)
# ============================================================================
print("\n" + "🟡"*35)
print("📍 STAGE 2: Fine-Tuning (Partial Base Unfreezing)")
print("🟡"*35)

# Unfreeze top layers of base model
base_model.trainable = True

# Freeze first 120 layers (keep low-level features frozen)
for layer in base_model.layers[:120]:
    layer.trainable = False

trainable_count = sum([1 for layer in model.layers if layer.trainable])

print(f"\n⚙️  Configuration:")
print(f"   Epochs: 30")
print(f"   Learning rate: 0.0001 (10x lower)")
print(f"   Base model: PARTIALLY UNFROZEN")
print(f"   Frozen layers: First 120 of {len(base_model.layers)}")
print(f"   Trainable layers: {trainable_count}")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_stage2 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        verbose=1,
        mode='max',
        min_delta=0.001
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=7,
        min_lr=0.000001,
        verbose=1,
        mode='max'
    ),
    keras.callbacks.ModelCheckpoint(
        'stage2_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

print("\n🏃 Training...")
start_time = datetime.now()

history_stage2 = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=30,
    callbacks=callbacks_stage2,
    class_weight=class_weights,
    verbose=1
)

elapsed = datetime.now() - start_time
print(f"\n⏱️  Stage 2 completed in {elapsed}")

best_val_acc_s2 = max(history_stage2.history['val_accuracy']) * 100
final_train_acc_s2 = history_stage2.history['accuracy'][-1] * 100
final_val_acc_s2 = history_stage2.history['val_accuracy'][-1] * 100

print(f"\n📊 Stage 2 Results:")
print(f"   Best validation accuracy: {best_val_acc_s2:.2f}%")
print(f"   Final training accuracy: {final_train_acc_s2:.2f}%")
print(f"   Final validation accuracy: {final_val_acc_s2:.2f}%")
print(f"   Overfitting gap: {abs(final_train_acc_s2 - final_val_acc_s2):.2f}%")

all_histories.append(history_stage2)

# ============================================================================
# 🎯 STAGE 3: Full Fine-Tuning (All Layers Unfrozen)
# ============================================================================
print("\n" + "🟢"*35)
print("📍 STAGE 3: Full Fine-Tuning (All Layers Unfrozen)")
print("🟢"*35)

# Unfreeze all layers but keep first 80 frozen
for layer in base_model.layers[:80]:
    layer.trainable = False
    
for layer in base_model.layers[80:]:
    layer.trainable = True

trainable_count = sum([1 for layer in model.layers if layer.trainable])

print(f"\n⚙️  Configuration:")
print(f"   Epochs: 40")
print(f"   Learning rate: 0.00005 (20x lower than initial)")
print(f"   Base model: MOSTLY UNFROZEN")
print(f"   Frozen layers: First 80 of {len(base_model.layers)}")
print(f"   Trainable layers: {trainable_count}")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.00005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_stage3 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=20,
        restore_best_weights=True,
        verbose=1,
        mode='max',
        min_delta=0.0005
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=10,
        min_lr=0.0000001,
        verbose=1,
        mode='max'
    ),
    keras.callbacks.ModelCheckpoint(
        'stage3_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

print("\n🏃 Training...")
start_time = datetime.now()

history_stage3 = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=40,
    callbacks=callbacks_stage3,
    class_weight=class_weights,
    verbose=1
)

elapsed = datetime.now() - start_time
print(f"\n⏱️  Stage 3 completed in {elapsed}")

best_val_acc_s3 = max(history_stage3.history['val_accuracy']) * 100
final_train_acc_s3 = history_stage3.history['accuracy'][-1] * 100
final_val_acc_s3 = history_stage3.history['val_accuracy'][-1] * 100

print(f"\n📊 Stage 3 Results:")
print(f"   Best validation accuracy: {best_val_acc_s3:.2f}%")
print(f"   Final training accuracy: {final_train_acc_s3:.2f}%")
print(f"   Final validation accuracy: {final_val_acc_s3:.2f}%")
print(f"   Overfitting gap: {abs(final_train_acc_s3 - final_val_acc_s3):.2f}%")

all_histories.append(history_stage3)

# ============================================================================
# 📊 FINAL SUMMARY
# ============================================================================
print("\n" + "="*70)
print("🏆 TRAINING COMPLETE!")
print("="*70)

# Find best validation accuracy across all stages
best_overall = max(best_val_acc_s1, best_val_acc_s2, best_val_acc_s3)

print(f"\n📊 FINAL RESULTS:")
print(f"   Stage 1 Best: {best_val_acc_s1:.2f}%")
print(f"   Stage 2 Best: {best_val_acc_s2:.2f}%")
print(f"   Stage 3 Best: {best_val_acc_s3:.2f}%")
print(f"\n   🏆 BEST VALIDATION ACCURACY: {best_overall:.2f}%")

if best_overall >= 90:
    print("\n🎉 EXCELLENT! Achieved 90%+ accuracy target!")
elif best_overall >= 85:
    print("\n✅ VERY GOOD! Close to 90% target")
elif best_overall >= 80:
    print("\n✅ GOOD! Model should work well in practice")
elif best_overall >= 70:
    print("\n⚠️  MODERATE. Model will work but may make mistakes")
else:
    print("\n⚠️  LOW ACCURACY. Possible causes:")
    print("   • Dataset quality issues (blurry images, wrong labels)")
    print("   • Disease classes look too similar")
    print("   • Need more images (target: 500+ per class)")
    print("   • Check if all images are correctly labeled")

# Combine all histories for plotting
history = type('obj', (object,), {
    'history': {}
})()

for key in ['accuracy', 'val_accuracy', 'loss', 'val_loss']:
    history.history[key] = []
    for h in all_histories:
        history.history[key].extend(h.history[key])

print("\n✅ Ready to visualize results and save model!")

## 📍 STEP 9: Visualize Training Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("📊 Generating training charts...\n")

# Create figure with 2 subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

epochs_range = range(1, len(history.history['accuracy']) + 1)

# Mark stage boundaries
stage1_end = len(all_histories[0].history['accuracy'])
stage2_end = stage1_end + len(all_histories[1].history['accuracy'])

# Plot 1: Accuracy
ax1.plot(epochs_range, history.history['accuracy'], 'b-', label='Training Accuracy', linewidth=2)
ax1.plot(epochs_range, history.history['val_accuracy'], 'r-', label='Validation Accuracy', linewidth=2)
ax1.axvline(x=stage1_end, color='gray', linestyle='--', alpha=0.5, label='Stage 1→2')
ax1.axvline(x=stage2_end, color='gray', linestyle='--', alpha=0.5, label='Stage 2→3')
ax1.axhline(y=0.90, color='green', linestyle=':', alpha=0.7, label='90% Target')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Model Accuracy (3-Stage Training)', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1])

# Plot 2: Loss
ax2.plot(epochs_range, history.history['loss'], 'b-', label='Training Loss', linewidth=2)
ax2.plot(epochs_range, history.history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
ax2.axvline(x=stage1_end, color='gray', linestyle='--', alpha=0.5, label='Stage 1→2')
ax2.axvline(x=stage2_end, color='gray', linestyle='--', alpha=0.5, label='Stage 2→3')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('Model Loss (3-Stage Training)', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
print("✅ Saved: training_history.png")
plt.show()

# Print stage-specific statistics
print("\n" + "="*70)
print("📈 TRAINING STATISTICS BY STAGE")
print("="*70)

for i, (h, name) in enumerate(zip(all_histories, ['Stage 1', 'Stage 2', 'Stage 3']), 1):
    print(f"\n{name}:")
    best_val = max(h.history['val_accuracy']) * 100
    best_epoch = h.history['val_accuracy'].index(max(h.history['val_accuracy'])) + 1
    final_train = h.history['accuracy'][-1] * 100
    final_val = h.history['val_accuracy'][-1] * 100
    
    print(f"   Best validation: {best_val:.2f}% (epoch {best_epoch})")
    print(f"   Final train/val: {final_train:.2f}% / {final_val:.2f}%")
    print(f"   Overfitting gap: {abs(final_train - final_val):.2f}%")

## 📍 STEP 10: Save Model & Class Names

In [ ]:
import os
import json
import shutil

print("="*70)
print("💾 SAVING MODEL FILES")
print("="*70)

# Create output directory
output_dir = '/content/trained_model'
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

# 1. Save the model in Keras format (.keras)
model_path = os.path.join(output_dir, 'black_pepper_disease_model.keras')
model.save(model_path)
print(f"\n✅ Saved model: {model_path}")

# 2. Save class names
class_names_data = {
    "class_names": class_names,
    "num_classes": num_classes,
    "class_indices": class_indices
}

class_names_path = os.path.join(output_dir, 'black_pepper_class_names.json')
with open(class_names_path, 'w') as f:
    json.dump(class_names_data, f, indent=2)
print(f"✅ Saved class names: {class_names_path}")

# 3. Save training summary
summary_data = {
    "model_name": "black_pepper_disease_model",
    "framework": "TensorFlow/Keras",
    "model_architecture": "MobileNetV2 + Custom Head",
    "training_strategy": "3-stage progressive training",
    "image_size": IMG_SIZE,
    "num_classes": num_classes,
    "class_names": class_names,
    "total_training_images": train_generator.samples,
    "total_validation_images": validation_generator.samples,
    "stage1_best_val_acc": float(best_val_acc_s1),
    "stage2_best_val_acc": float(best_val_acc_s2),
    "stage3_best_val_acc": float(best_val_acc_s3),
    "best_overall_val_acc": float(best_overall),
    "final_train_acc": float(final_train_acc_s3),
    "final_val_acc": float(final_val_acc_s3),
    "training_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

summary_path = os.path.join(output_dir, 'training_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary_data, f, indent=2)
print(f"✅ Saved training summary: {summary_path}")

# 4. Copy training chart
if os.path.exists('training_history.png'):
    chart_dest = os.path.join(output_dir, 'training_history.png')
    shutil.copy('training_history.png', chart_dest)
    print(f"✅ Saved training chart: {chart_dest}")

print("\n" + "="*70)
print("✅ ALL FILES SAVED!")
print("="*70)

print(f"\n📂 Output directory: {output_dir}")
print("\n📦 Files created:")
print("   • black_pepper_disease_model.keras     (Main model file)")
print("   • black_pepper_class_names.json        (Class labels)")
print("   • training_summary.json                (Training stats)")
print("   • training_history.png                 (Training chart)")

# Copy to Google Drive (optional backup)
drive_backup = '/content/drive/MyDrive/trained_black_pepper_model'
try:
    if os.path.exists(drive_backup):
        shutil.rmtree(drive_backup)
    shutil.copytree(output_dir, drive_backup)
    print(f"\n✅ Backup saved to Google Drive: {drive_backup}")
except Exception as e:
    print(f"\n⚠️  Could not backup to Drive: {e}")

print("\n🎉 Model is ready! Proceed to download files.")

## 📍 STEP 11: Test Model with Sample Predictions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

print("="*70)
print("🧪 TESTING MODEL WITH SAMPLE IMAGES")
print("="*70)

# Get some validation images to test
val_batch = next(validation_generator)
test_images, test_labels = val_batch

# Make predictions
predictions = model.predict(test_images[:9])

# Plot first 9 images with predictions
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle('Sample Predictions on Validation Images', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(test_images):
        # Display image
        ax.imshow(test_images[i])
        
        # Get true and predicted classes
        true_class_idx = np.argmax(test_labels[i])
        pred_class_idx = np.argmax(predictions[i])
        
        true_class = class_names[true_class_idx]
        pred_class = class_names[pred_class_idx]
        pred_confidence = predictions[i][pred_class_idx] * 100
        
        # Color code: green if correct, red if wrong
        color = 'green' if true_class_idx == pred_class_idx else 'red'
        
        # Set title
        title = f'True: {true_class}\nPred: {pred_class}\nConf: {pred_confidence:.1f}%'
        ax.set_title(title, fontsize=10, color=color, fontweight='bold')
        ax.axis('off')
    else:
        ax.axis('off')

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
print("\n✅ Saved: sample_predictions.png")
plt.show()

# Calculate accuracy on these samples
correct = sum([np.argmax(test_labels[i]) == np.argmax(predictions[i]) for i in range(len(test_images[:9]))])
print(f"\n📊 Accuracy on shown samples: {correct}/9 ({correct/9*100:.1f}%)")

print("\n✅ Testing complete!")

## 📍 STEP 12: Download Files to Your Computer

Run this cell to download all trained model files to your local computer.

In [ ]:
from google.colab import files
import os

print("="*70)
print("⬇️  DOWNLOADING FILES")
print("="*70)

output_dir = '/content/trained_model'

if not os.path.exists(output_dir):
    print("❌ Error: Output directory not found!")
    print("   Make sure you ran the 'Save Model' cell first")
else:
    print("\n📦 Downloading files...\n")
    
    # Download each file
    files_to_download = [
        'black_pepper_disease_model.keras',
        'black_pepper_class_names.json',
        'training_summary.json',
        'training_history.png',
        'sample_predictions.png'
    ]
    
    for filename in files_to_download:
        filepath = os.path.join(output_dir, filename)
        if os.path.exists(filepath):
            print(f"Downloading: {filename}")
            files.download(filepath)
        else:
            print(f"⚠️  Not found: {filename}")
    
    print("\n" + "="*70)
    print("✅ DOWNLOAD COMPLETE!")
    print("="*70)
    
    print("\n📝 NEXT STEPS:")
    print("\n1. Find the downloaded files in your Downloads folder")
    print("\n2. Copy these files to your local API:")
    print("   C:\\xampp\\htdocs\\PEPPER\\backend\\python\\models\\")
    print("\n3. Replace the old model files with the new ones")
    print("\n4. Restart your Python API (port 5001)")
    print("\n5. Test with black pepper leaf images!")
    
    print("\n🎉 Your model is ready to use!")

---

## 🎉 Training Complete!

### What You Should Have:
✅ Trained CNN model file (`.keras`)  
✅ Class names JSON file  
✅ Training summary with accuracy stats  
✅ Training history charts  
✅ Sample predictions visualization  

### Expected Accuracy:
- **90%+**: Excellent! Model ready for production
- **85-90%**: Very good! Model will work well
- **80-85%**: Good! Usable for most cases
- **70-80%**: Moderate. May need more training data
- **<70%**: Consider improving dataset quality

### If Accuracy is Below 90%:

**Possible Causes:**
1. **Dataset issues**: Blurry images, wrong labels, poor lighting
2. **Not enough images**: Target 500+ images per class
3. **Classes too similar**: Diseases look too alike
4. **Image quality**: Ensure clear, well-lit leaf photos

**Solutions:**
- Collect more diverse images (different angles, lighting, backgrounds)
- Verify all images are correctly labeled
- Ensure images are clear and show disease symptoms clearly
- Consider removing ambiguous images
- Try training longer (more epochs in Stage 3)

### For Production Deployment:

Copy these files to your local system:
```
C:\xampp\htdocs\PEPPER\backend\python\models\
  ├── black_pepper_disease_model.keras
  └── black_pepper_class_names.json
```

Then restart your Python API and test!

---

**Need Help?** Check `training_summary.json` for detailed stats about your training session.